# Data budget

A worked example of the `quicksat` data and downlink budget against the sample satellite: how much the payload generates, how much the downlink clears, and whether the second keeps up with the first.

For *why* each step works the way it does — the byte convention, the unit traps, why the input is config rather than a CSV — see `docs/data_budget_ref.ipynb`. For the orbit the budget sits on, see `docs/orbit_ref.ipynb`.

In [1]:
# Change log. LAST_CHANGE and CHANGE_NOTE are typed in by hand: update them whenever
# the inputs move -- a duty cycle revised, a contact count renegotiated -- so that the
# figures below can be read against what produced them. The run time is recorded
# automatically, and says how stale the outputs stored in this notebook are relative
# to that last change.
from datetime import datetime

LAST_CHANGE = "2026-09-16"
CHANGE_NOTE = "Raw data rate cut to 1000 Mbit/s; budget now closes at +12%."

print(f"last change  {LAST_CHANGE}")
print(f"             {CHANGE_NOTE}")
print(f"last run     {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

last change  2026-09-16
             Raw data rate cut to 1000 Mbit/s; budget now closes at +12%.
last run     2026-09-21 00:43 CEST


In [2]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from sample/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat.dataflow.budget import DataBudget, DataFlowModel
from quicksat.utils.orbit import Orbit

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## The inputs

Two files: the shared orbit, and the payload dataflow model. The orbit is loaded separately because the delta-V budget reads the same one, and agility will — loading it once here and handing it over keeps a single object rather than one parse per budget. Where nothing else needs the orbit, `DataBudget.from_yaml_file()` takes the two paths and reads both files itself.

In [3]:
DATA = Path("sample") / "data"
orbit = Orbit.from_yaml_file(DATA / "orbit.yaml")
model = DataFlowModel.from_yaml_file(DATA / "pl_dataflow_model.yaml")

data_budget = DataBudget(model, orbit)

In [4]:
gen = data_budget.model.generation
dl = data_budget.model.downlink

print(f"orbit        {orbit.altitude:~.0f} circular, {orbit.inclination:~}")
print(f"             period {orbit.period:~.0f}, {orbit.orbits_per_day:~.3f} orbits/day")
print(f"generation   {gen.raw_datarate:~.0f} raw, {gen.compression_ratio}x compression,"
      f" {gen.duty_cycle:~} duty cycle")
print(f"downlink     {dl.rate:~.0f}, {dl.contacts_per_day:g} contacts/day"
      f" at {dl.avg_contact_duration:~.0f}")

orbit        500 km circular, 97.4 deg
             period 5677 s, 15.219 orbits/day
generation   1000 Mbit / s raw, 2.4x compression, 5 % duty cycle
downlink     800 Mbit / s, 7 contacts/day at 6 min


## Generation and downlink

Generation is per orbit — the payload collects for a fraction of each revolution. Downlink is per day, because contacts are counted against the ground station's day. The two meet at the daily figure.

In [5]:
print(f"effective rate    {data_budget.effective_datarate:~.2f}  "
      f"({gen.raw_datarate:~.0f} raw / {gen.compression_ratio}x)")
print(f"collecting for    {data_budget.generation_duration:~.1f} of every {orbit.period:~.0f}")
print()
print(f"generated         {data_budget.generated_per_orbit:~.2f} per orbit")
print(f"                  {data_budget.generated_per_day:~.2f} per day")
print()
print(f"contacts          {dl.contacts_per_day:g} per day at {dl.avg_contact_duration:~.0f}"
      f"  =  {data_budget.contact_per_day:~.0f} per day")
print(f"downlinked        {data_budget.downlinked_per_day:~.2f} per day")
print(f"                  {data_budget.downlinked_per_orbit:~.2f} per orbit")

effective rate    416.67 Mbit / s  (1000 Mbit / s raw / 2.4x)
collecting for    283.8 s of every 5677 s

generated         14.78 GB per orbit
                  225.00 GB per day

contacts          7 per day at 6 min  =  42 min per day
downlinked        252.00 GB per day
                  16.56 GB per orbit


## Does it close?

The one number the budget exists to produce.

In [6]:
margin = data_budget.margin.magnitude
balance = data_budget.downlinked_per_day - data_budget.generated_per_day

print(f"generated    {data_budget.generated_per_day:~.2f} per day")
print(f"downlinked   {data_budget.downlinked_per_day:~.2f} per day")
print(f"margin       {margin * 100:+.1f}%")
print()
if margin >= 0:
    print(f"CLOSES with {balance:~.2f} per day to spare")
else:
    print(f"DOES NOT CLOSE - short by {abs(balance):~.2f} per day")
    print(f"needs {1 / (1 + margin):.2f}x the current downlink capacity")

generated    225.00 GB per day
downlinked   252.00 GB per day
margin       +12.0%

CLOSES with 27.00 GB per day to spare


### How much headroom is there?

A margin is only as good as the assumptions under it, and every input here is an estimate. The useful question is not "what is the margin" but "how far can each input move before there isn't one".

Each row below is the **break-even value** for one input, holding everything else at what the files say. Read the duty cycle row as: the payload could collect up to that fraction of each orbit before the downlink stops keeping up.

In [7]:
headroom = 1 + margin

pd.DataFrame(
    [
        ("raw data rate", f"{gen.raw_datarate:~.0f}",
         f"{(gen.raw_datarate * headroom).to('Mbit/s'):~.0f}", "generate faster"),
        ("duty cycle", f"{gen.duty_cycle:~}",
         f"{gen.duty_cycle * headroom:~.2f}", "collect longer"),
        ("compression ratio", f"{gen.compression_ratio}x",
         f"{gen.compression_ratio / headroom:.2f}x", "compress less"),
        ("contacts per day", f"{dl.contacts_per_day:g}",
         f"{dl.contacts_per_day / headroom:.2f}", "fewer passes"),
        ("contact duration", f"{dl.avg_contact_duration:~.0f}",
         f"{(dl.avg_contact_duration / headroom).to('min'):~.2f}", "shorter passes"),
        ("downlink rate", f"{dl.rate:~.0f}",
         f"{(dl.rate / headroom).to('Mbit/s'):~.0f}", "slower link"),
    ],
    columns=["input", "now", "break-even", "moving which way"],
)

,input,now,break-even,moving which way
0,raw data rate,1000 Mbit / s,1120 Mbit / s,generate faster
1,duty cycle,5 %,5.60 %,collect longer
2,compression ratio,2.4x,2.14x,compress less
3,contacts per day,7,6.25,fewer passes
4,contact duration,6 min,5.36 min,shorter passes
5,downlink rate,800 Mbit / s,714 Mbit / s,slower link


## Storage

What has to be held on board across a run of orbits with no usable contact, against what is actually flying.

In [8]:
orbits = data_budget.model.storage.orbits_without_contact
print(f"{orbits:g} orbits without contact  ->  {data_budget.storage_required():~.2f}")
print(f"1 orbit                   ->  {data_budget.storage_required(1):~.2f}")
print(f"a full day                ->  {data_budget.storage_required(orbit.orbits_per_day.magnitude):~.2f}")

equipment = pd.read_csv(DATA / "equipment.csv")
ssdr = equipment.query("equipment_id == 'mass_memory'").iloc[0]
print(f"\nflying: {ssdr['equipment_name']}")

3 orbits without contact  ->  44.35 GB
1 orbit                   ->  14.78 GB
a full day                ->  225.00 GB



flying: SSDR 2 TB


## The budget, in one table

`tabulated_data()` renders the whole chain as a document: item, value, unit, and the assumption behind it. The frame behind it is available as `.data`, with every row tagged `input`, `derived`, `margin` or `storage`.

In [9]:
data_budget.tabulated_data()

,Value,Unit,Comment
Orbital period,94.62,min,500 km circular
Orbits per day,15.22,-,
Data generation rate,416.67,Mbit/s,"1000 Mbit / s raw, 2.4x compression"
Data generation duration,283.85,s/orbit,5 % duty cycle
Data generated,14.78,GB/orbit,
,225.00,GB/day,
Data downlink rate,800.00,Mbit/s,achieved throughput
Contacts,7.00,per day,6 min average duration
Data downlink duration,42.00,min/day,
Data downlinked,16.56,GB/orbit,
